# Grok-multimodal · FS07-FS08 Document intelligence

DocQA via OCR+TF-IDF, then visual page-patch retrieval.


In [ ]:
import os, json, math, random, time, re, string
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path("/kaggle/working"); FIG=OUT/"figures"; RES=OUT/"results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device",device,"gpus",torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}


## FS07 · DocQA


In [ ]:
# Mini document: render text as image-like "page", OCR = known text (oracle) + noise OCR, then retrieve+answer
DOCS = [
    {"id":"d1","title":"Invoice A","text":"Invoice number 1001. Total amount due is 42 dollars. Vendor is Acme Corp. Date 2024-01-15."},
    {"id":"d2","title":"Invoice B","text":"Invoice number 1002. Total amount due is 99 dollars. Vendor is Globex. Date 2024-02-20."},
    {"id":"d3","title":"Report","text":"Quarterly report. Revenue grew 12 percent. Headcount is 350. Office is Tokyo."},
    {"id":"d4","title":"Manual","text":"Safety manual. Wear gloves. Max temperature 80 celsius. Emergency exit is north."},
]

def render_doc_image(doc, W=256, H=128):
    """Rasterize text lines onto a white canvas (fake page image)."""
    img=np.ones((H,W,3),np.float32)
    # draw header bar
    img[:18,:]=0.85
    fig=plt.figure(figsize=(W/64,H/64), dpi=64)
    ax=fig.add_axes([0,0,1,1]); ax.imshow(img); ax.set_xlim(0,W); ax.set_ylim(H,0); ax.axis("off")
    ax.text(8,12, doc["title"], fontsize=8, color="black", va="center")
    # wrap text
    words=doc["text"].split(); lines=[]; cur=""
    for w in words:
        if len(cur)+len(w)+1>40: lines.append(cur); cur=w
        else: cur=(cur+" "+w).strip()
    if cur: lines.append(cur)
    y=30
    for line in lines[:6]:
        ax.text(8,y,line,fontsize=6,color="black",va="top"); y+=14
    fig.canvas.draw()
    # grab rgb
    buf=np.asarray(fig.canvas.buffer_rgba())[:,:,:3].astype(np.float32)/255.0
    plt.close(fig)
    # resize-ish
    return buf

def fake_ocr(doc, noise=0.0):
    """Oracle OCR with optional char corruption."""
    t=doc["text"]
    if noise<=0: return t
    chars=list(t)
    for i in range(len(chars)):
        if chars[i].isalnum() and random.random()<noise:
            chars[i]=random.choice(string.ascii_lowercase)
    return "".join(chars)

# TF-IDF bag retrieval
def tokenize(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def build_tfidf(docs_texts):
    df=Counter(); tfs=[]
    for t in docs_texts:
        toks=tokenize(t); tf=Counter(toks); tfs.append(tf)
        for w in set(toks): df[w]+=1
    N=len(docs_texts); vocab=sorted(df.keys())
    idf={w:math.log((N+1)/(df[w]+1))+1 for w in vocab}
    vecs=[]
    for tf in tfs:
        v=np.array([tf[w]*idf[w] for w in vocab],np.float32)
        v=v/(np.linalg.norm(v)+1e-8); vecs.append(v)
    return vocab, idf, np.stack(vecs)

ocr_texts=[fake_ocr(d,0.0) for d in DOCS]
vocab,idf,doc_vecs=build_tfidf(ocr_texts)

def encode_query(q):
    tf=Counter(tokenize(q))
    v=np.array([tf[w]*idf.get(w,0) for w in vocab],np.float32)
    return v/(np.linalg.norm(v)+1e-8)

def retrieve(q, k=2):
    qv=encode_query(q)
    sims=doc_vecs@qv
    order=np.argsort(-sims)
    return [(DOCS[i]["id"], float(sims[i]), DOCS[i]) for i in order[:k]]

def answer_span(question, doc_text):
    """Very small extractive QA: find number/entity by question cues."""
    ql=question.lower(); toks=doc_text.split()
    if "invoice number" in ql or "invoice" in ql and "number" in ql:
        m=re.search(r"Invoice number (\d+)", doc_text); return m.group(1) if m else "?"
    if "amount" in ql or "total" in ql or "dollars" in ql:
        m=re.search(r"(\d+) dollars", doc_text); return m.group(1)+" dollars" if m else "?"
    if "vendor" in ql:
        m=re.search(r"Vendor is ([A-Za-z ]+)\.", doc_text); return m.group(1).strip() if m else "?"
    if "headcount" in ql:
        m=re.search(r"Headcount is (\d+)", doc_text); return m.group(1) if m else "?"
    if "temperature" in ql:
        m=re.search(r"(\d+) celsius", doc_text); return m.group(1)+" celsius" if m else "?"
    if "office" in ql:
        m=re.search(r"Office is ([A-Za-z]+)", doc_text); return m.group(1) if m else "?"
    # fallback: longest noun-ish token
    return toks[min(3,len(toks)-1)]

QA_SET=[
    ("What is the invoice number for Acme?","1001","d1"),
    ("How many dollars does Globex invoice?","99 dollars","d2"),
    ("What is the headcount?","350","d3"),
    ("What is max temperature?","80 celsius","d4"),
    ("Who is the vendor on invoice 1001?","Acme Corp","d1"),
]

rows7=[]
for q,gt,gt_doc in QA_SET:
    hits=retrieve(q,k=2)
    top_id,score,doc=hits[0]
    ans=answer_span(q, doc["text"])
    rows7.append({"q":q,"gt":gt,"gt_doc":gt_doc,"ret_doc":top_id,"ret_ok":top_id==gt_doc,
                  "ans":ans,"ans_ok":ans.lower()==gt.lower() or gt.lower() in ans.lower()})

# visualize pages
fig,axes=plt.subplots(2,2,figsize=(8,5))
for ax,d in zip(axes.ravel(), DOCS):
    page=render_doc_image(d)
    ax.imshow(page); ax.set_title(d["id"]+": "+d["title"],fontsize=9); ax.axis("off")
fig.suptitle("FS07 synthetic document pages"); fig.tight_layout()
fig.savefig(FIG/"fs07_pages.png",dpi=120); plt.close()

ret_acc=sum(r["ret_ok"] for r in rows7)/len(rows7)
ans_acc=sum(r["ans_ok"] for r in rows7)/len(rows7)
print(rows7, "ret",ret_acc,"ans",ans_acc)
fs07={"stage":"FS07","method":"page render + OCR text + TF-IDF retrieve + span answer",
      "retrieval_acc":ret_acc,"answer_acc":ans_acc,"rows":rows7,
      "vs_prev":"FS06 answers about shapes; FS07 answers from multi-page text documents",
      "figure":"figures/fs07_pages.png"}
(RES/"fs07.json").write_text(json.dumps(fs07,indent=2)); PROGRESS["FS07"]="ok"; print("FS07 DONE")


## FS08 · Visual document retrieval


In [ ]:
# Visual document retrieval: embed page image patches (ColPali intuition) vs text-only TF-IDF
# Split each page into grid patches, embed with tiny CNN, maxsim against query token embeds

class PatchEnc(nn.Module):
    def __init__(self,d=32):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.ReLU(),
            nn.Conv2d(16,d,3,padding=1),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten())
    def forward(self, patches):  # [B*N,3,h,w]
        return F.normalize(self.cnn(patches),dim=-1)

class QEnc(nn.Module):
    def __init__(self,vmax,d=32):
        super().__init__()
        self.emb=nn.Embedding(vmax,d)
    def forward(self, ids):  # [B,L]
        return F.normalize(self.emb(ids),dim=-1)

def page_to_patches(img, gh=4, gw=4, ph=32, pw=32):
    H,W,_=img.shape
    patches=[]
    for i in range(gh):
        for j in range(gw):
            y0=int(i*H/gh); x0=int(j*W/gw)
            y1=int((i+1)*H/gh); x1=int((j+1)*W/gw)
            crop=img[y0:y1,x0:x1]
            # nearest resize
            ys=(np.linspace(0,crop.shape[0]-1,ph)).astype(int)
            xs=(np.linspace(0,crop.shape[1]-1,pw)).astype(int)
            r=crop[ys][:,xs]
            patches.append(r.transpose(2,0,1))
    return np.stack(patches)  # [N,3,ph,pw]

# build query vocab from questions + doc words
all_toks=set()
for d in DOCS: all_toks.update(tokenize(d["text"])); all_toks.update(tokenize(d["title"]))
for q,_,_, in [(r["q"],None,None) for r in rows7]:
    all_toks.update(tokenize(q))
# also QA_SET
for q,_,_ in QA_SET: all_toks.update(tokenize(q))
vv=["<pad>"]+sorted(all_toks)
vstoi={t:i for i,t in enumerate(vv)}

def qids(q,L=8):
    ids=[vstoi.get(t,0) for t in tokenize(q)][:L]
    ids+=[0]*(L-len(ids)); return ids

# train: match page patches to its own title/text tokens (weak)
penc=PatchEnc().to(device); qenc=QEnc(len(vv)).to(device)
opt=torch.optim.Adam(list(penc.parameters())+list(qenc.parameters()), lr=2e-3)

pages=[render_doc_image(d) for d in DOCS]
patch_bank=[page_to_patches(p) for p in pages]

def maxsim(q_tok_emb, patch_emb):
    # q: [L,d], p:[N,d] -> sum_i max_j q_i·p_j
    sim=q_tok_emb@patch_emb.t()  # [L,N]
    return sim.max(dim=1).values.sum()

hist8=[]
for epoch in range(1,26):
    penc.train(); qenc.train(); losses=[]
    for di,d in enumerate(DOCS):
        patches=torch.tensor(patch_bank[di],dtype=torch.float32,device=device)
        pe=penc(patches)  # [N,d]
        # positive query from doc text snippet
        q=" ".join(tokenize(d["title"]+" "+d["text"])[:8])
        ids=torch.tensor([qids(q)],device=device)
        qe=qenc(ids)[0]
        # in-batch negatives: other docs
        scores=[]
        for dj in range(len(DOCS)):
            pj=torch.tensor(patch_bank[dj],dtype=torch.float32,device=device)
            scores.append(maxsim(qe, penc(pj)))
        scores=torch.stack(scores)
        loss=F.cross_entropy(scores.unsqueeze(0), torch.tensor([di],device=device))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
    hist8.append({"epoch":epoch,"loss":round(float(np.mean(losses)),4)})
    if epoch%5==0: print(hist8[-1])

# eval retrieval for QA_SET questions
penc.eval(); qenc.eval()
rows8=[]
with torch.no_grad():
    pe_all=[penc(torch.tensor(pb,dtype=torch.float32,device=device)) for pb in patch_bank]
    for q,gt,gt_doc in QA_SET:
        qe=qenc(torch.tensor([qids(q)],device=device))[0]
        sims=[float(maxsim(qe,pe).cpu()) for pe in pe_all]
        top=int(np.argmax(sims))
        rows8.append({"q":q,"gt_doc":gt_doc,"vis_ret":DOCS[top]["id"],"ok":DOCS[top]["id"]==gt_doc,
                      "sims":[round(s,3) for s in sims]})
vis_acc=sum(r["ok"] for r in rows8)/len(rows8)
# compare to text TF-IDF from FS07
text_acc=sum(1 for q,_,gd in QA_SET if retrieve(q,1)[0][0]==gd)/len(QA_SET)
print("visual R@1",vis_acc,"text R@1",text_acc,rows8)

# viz patches for one page
fig,axes=plt.subplots(4,4,figsize=(6,6))
p0=patch_bank[0]
for i,ax in enumerate(axes.ravel()):
    ax.imshow(p0[i].transpose(1,2,0)); ax.axis("off")
fig.suptitle("FS08 page patches (ColPali intuition)")
fig.tight_layout(); fig.savefig(FIG/"fs08_patches.png",dpi=120); plt.close()

fs08={"stage":"FS08","method":"patch embeddings + MaxSim (ColPali-style visual doc retrieval)",
      "visual_R@1":vis_acc,"text_tfidf_R@1":text_acc,"rows":rows8,"history":hist8,
      "vs_prev":"FS07 retrieves on OCR text only; FS08 retrieves from page pixels/patches",
      "figure":"figures/fs08_patches.png"}
(RES/"fs08.json").write_text(json.dumps(fs08,indent=2)); PROGRESS["FS08"]="ok"; print("FS08 DONE")


In [ ]:
summary={"notebook":"Grok-multimodal-fs07-fs08-doc","progress":PROGRESS,"device":str(device)}
(RES/"summary_fs07_fs08.json").write_text(json.dumps(summary,indent=2))
(OUT/"SUCCESS").write_text("ok\n"); print(summary)
